# Brain Tumor Segmentation using U-Net
This notebook uses synthetic data to simulate brain tumor segmentation using a U-Net-based model.

In [ ]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, UpSampling2D, Concatenate
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split


In [ ]:
IMG_SIZE = 128
BATCH_SIZE = 32
EPOCHS = 10
MODEL_NAME = "tumor_segmentation_model.keras"

class TumorSegmenter:
    def __init__(self):
        self.model = None
        self.sample_images = self._generate_sample_data()
        
    def _generate_sample_data(self):
        images = []
        masks = []
        for i in range(10):
            img = np.random.rand(IMG_SIZE, IMG_SIZE, 3) * 0.5
            center = (IMG_SIZE//2, IMG_SIZE//2)
            radius = IMG_SIZE//4
            y, x = np.ogrid[:IMG_SIZE, :IMG_SIZE]
            mask = (x - center[0])**2 + (y - center[1])**2 <= radius**2
            img[mask] += 0.5
            mask = mask.astype(float).reshape(IMG_SIZE, IMG_SIZE, 1)
            images.append(img)
            masks.append(mask)
        return np.array(images), np.array(masks)
    
    def load_data(self):
        images, masks = self.sample_images
        masked_images = images * masks
        X = np.concatenate((images, masked_images), axis=-1)
        return train_test_split(X, masks, test_size=0.2, random_state=42)
    
    def build_model(self):
        inputs = Input((IMG_SIZE, IMG_SIZE, 6))
        conv1 = Conv2D(32, (3, 3), activation='relu', padding='same')(inputs)
        pool1 = MaxPooling2D((2, 2))(conv1)
        conv2 = Conv2D(64, (3, 3), activation='relu', padding='same')(pool1)
        pool2 = MaxPooling2D((2, 2))(conv2)
        conv3 = Conv2D(128, (3, 3), activation='relu', padding='same')(pool2)
        up1 = UpSampling2D((2, 2))(conv3)
        concat1 = Concatenate()([conv2, up1])
        conv4 = Conv2D(64, (3, 3), activation='relu', padding='same')(concat1)
        up2 = UpSampling2D((2, 2))(conv4)
        concat2 = Concatenate()([conv1, up2])
        conv5 = Conv2D(1, (1, 1), activation='sigmoid')(concat2)
        model = Model(inputs=inputs, outputs=conv5)
        model.compile(optimizer=Adam(1e-4), loss='binary_crossentropy', metrics=['accuracy'])
        return model
    
    def train(self):
        X_train, X_test, y_train, y_test = self.load_data()
        self.model = self.build_model()
        history = self.model.fit(X_train, y_train, batch_size=BATCH_SIZE, epochs=EPOCHS, validation_data=(X_test, y_test))
        self.model.save(MODEL_NAME)
        return history


In [ ]:
segmenter = TumorSegmenter()
history = segmenter.train()


In [ ]:
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Model Accuracy over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)
plt.show()

plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Model Loss over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.show()
